# ⚡ AETHERION
## Aerial Emergency Threat and Hazard Response Intelligence Operations Network

> *"Guardian of the skies. Protector of lives."*

**Developed by P. Shiva Charan Reddy** · Personal Project · Hyderabad 🇮🇳 · 2026

---

**11 threat classes:** normal_swim, panic_drowning, unconscious, submerged, shark_attack, rip_current, jellyfish_swarm, heatstroke, fight_assault, net_entrapment, monsoon_surge

**Alert tiers:** CRITICAL (PS + Hospital + Coast Guard) → HIGH (Lifeguards) → MEDIUM → LOW (Public)

---

> ⚠️ **Runtime → Change runtime type → T4 GPU before running!**

## ✅ Step 1: Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')

In [ ]:
!pip install ultralytics roboflow supervision albumentations twilio firebase-admin -q
print('✅ Dependencies installed')

## 📦 Step 2: Dataset Setup

You need data for **11 classes**. Use multiple Roboflow datasets and merge them.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_KEY_HERE'  # Get free at roboflow.com

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset 1: Drowning detection (panic, unconscious, normal)
dataset1 = rf.workspace('drowning-detection-project').project('drowning-detection').version(1).download('yolov8')

# Dataset 2: Beach safety / crowd (fights, heatstroke)
# dataset2 = rf.workspace('beach-safety').project('beach-threats').version(1).download('yolov8')

DATASET_PATH = dataset1.location
print(f'Dataset: {DATASET_PATH}')

In [ ]:
# Clone reference repo for sample data
!git clone https://github.com/Hasibwajid/Automated-Drowning-Detection-YOLOV8 /content/ref -q
print('✅ Reference repo cloned')

## ⚙️ Step 3: Dataset YAML — 11 Classes

In [ ]:
import yaml, os

ALL_CLASSES = [
    'normal_swim',    # 0 - Regular swimming
    'panic_drowning', # 1 - Struggling, arms up, no progress  
    'unconscious',    # 2 - Face-down, motionless >15s
    'submerged',      # 3 - No head visible >15s
    'shark_attack',   # 4 - Thrashing + blood signature
    'rip_current',    # 5 - Cluster drifting seaward
    'jellyfish_swarm',# 6 - Surface clusters + reactions
    'heatstroke',     # 7 - Collapsed on beach, motionless
    'fight_assault',  # 8 - Confrontation, staggering
    'net_entrapment', # 9 - Irregular struggle near boat/nets
    'monsoon_surge',  # 10 - High wave pattern
]

dataset_yaml = {
    'path':  DATASET_PATH,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    len(ALL_CLASSES),
    'names': ALL_CLASSES,
}

yaml_path = '/content/dataset_v2.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f)
print('✅ Dataset YAML:')
print(open(yaml_path).read())

## 🏋️ Step 4: Train YOLOv8s — 11 Classes

In [ ]:
from ultralytics import YOLO

# YOLOv8s = best speed/accuracy for edge deployment
model = YOLO('yolov8s.pt')
print('Model loaded.')

In [ ]:
results = model.train(
    data        = yaml_path,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    name        = 'aetherion',
    project     = '/content/runs',
    patience    = 20,
    optimizer   = 'AdamW',
    lr0         = 0.001,

    # ── AQUATIC / BEACH AUGMENTATION ──────────────
    hsv_h       = 0.02,    # water hue variation
    hsv_s       = 0.75,    # saturation
    hsv_v       = 0.45,    # brightness (glare sim)
    fliplr      = 0.5,     # horizontal flip
    flipud      = 0.1,     # vertical flip (aerial)
    mosaic      = 1.0,     # mosaic augmentation
    degrees     = 12.0,    # drone angle variation
    translate   = 0.12,
    scale       = 0.55,
    shear       = 2.5,
    perspective = 0.0005,  # aerial perspective

    device      = 0,       # T4 GPU
    workers     = 2,
    save        = True,
    save_period = 10,
    verbose     = True,
)

print('\n✅ Training complete!')
print(f'Best model: /content/runs/aetherion/weights/best.pt')

## 📊 Step 5: Evaluate

In [ ]:
best = YOLO('/content/runs/aetherion/weights/best.pt')
metrics = best.val(data=yaml_path, split='test')

print('\n📊 EVALUATION RESULTS:')
print(f'  mAP@0.5:       {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95:  {metrics.box.map:.4f}')
print(f'  Precision:     {metrics.box.mp:.4f}')
print(f'  Recall:        {metrics.box.mr:.4f}')

# Per-class AP
if hasattr(metrics.box, 'ap_class_index'):
    print('\n  Per-class AP@0.5:')
    for i, cls in enumerate(ALL_CLASSES):
        try:
            print(f'    {cls:<20} {metrics.box.ap50[i]:.4f}')
        except:
            pass

In [ ]:
# Visualize results
from IPython.display import Image, display
import os

for plot in ['/content/runs/aetherion/results.png',
             '/content/runs/aetherion/confusion_matrix.png',
             '/content/runs/aetherion/val_batch0_pred.jpg']:
    if os.path.exists(plot):
        print(f'\n{os.path.basename(plot)}')
        display(Image(plot, width=800))

## 🎯 Step 6: Inference + Alert System Test

In [ ]:
import cv2, numpy as np, time

CLASS_NAMES = ALL_CLASSES
SEVERITY_MAP = {
    'normal_swim': 'LOW',
    'panic_drowning': 'HIGH',
    'unconscious': 'CRITICAL',
    'submerged': 'CRITICAL',
    'shark_attack': 'CRITICAL',
    'rip_current': 'MEDIUM',
    'jellyfish_swarm': 'LOW',
    'heatstroke': 'MEDIUM',
    'fight_assault': 'MEDIUM',
    'net_entrapment': 'HIGH',
    'monsoon_surge': 'MEDIUM',
}

RESPONDERS_MAP = {
    'CRITICAL': ['Police Station', 'Hospital (Ambulance)', 'Coast Guard'],
    'HIGH':     ['Lifeguard Team'],
    'MEDIUM':   ['Lifeguard Team', 'Public Alert'],
    'LOW':      ['Public Alert'],
}

def run_demo_inference(source, max_frames=100):
    cap = cv2.VideoCapture(source)
    frame_times = []
    
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret: break
        
        t = time.time()
        results = best(frame, conf=0.5, verbose=False)[0]
        lat = (time.time()-t)*1000
        frame_times.append(lat)
        
        for box in results.boxes:
            cls = CLASS_NAMES[int(box.cls[0])]
            conf = float(box.conf[0])
            sev = SEVERITY_MAP.get(cls, 'LOW')
            resp = RESPONDERS_MAP.get(sev, [])
            
            print(f'  [{sev}] {cls:<20} conf={conf:.2f}')
            if sev in ('CRITICAL', 'HIGH'):
                print(f'    → Alerting: {", ".join(resp)}')
                print(f'    → GPS: 17.4239°N, 78.4738°E (Hussain Sagar)')
    
    cap.release()
    print(f'\nAvg latency: {np.mean(frame_times):.1f}ms | FPS: {1000/np.mean(frame_times):.1f}')

# Upload video and run:
# run_demo_inference('/content/your_video.mp4')
print('Function ready. Upload a video and call run_demo_inference(path)')

## 🚨 Step 7: Twilio Alert Test

In [ ]:
# Optional: Test real SMS alerts
!pip install twilio -q
from twilio.rest import Client

TWILIO_SID   = 'YOUR_SID'
TWILIO_TOKEN = 'YOUR_TOKEN'
TWILIO_FROM  = '+1XXXXXXXXXX'

def send_tiered_alert(threat_class, confidence, lat=17.4239, lon=78.4738):
    sev = SEVERITY_MAP.get(threat_class, 'LOW')
    resp = RESPONDERS_MAP.get(sev, [])
    
    client = Client(TWILIO_SID, TWILIO_TOKEN)
    maps   = f'https://maps.google.com/?q={lat},{lon}'
    
    if sev == 'CRITICAL':
        msg = (
            f'🚨 CRITICAL EMERGENCY — AETHERION Beach Safety\n'
            f'Threat: {threat_class.upper()}\n'
            f'Confidence: {confidence*100:.1f}%\n'
            f'Location: {lat:.4f}°N, {lon:.4f}°E\n'
            f'Maps: {maps}\n'
            f'Notifying: Police Station + Hospital (Ambulance) + Coast Guard\n'
            f'Drone arriving with buoy.'
        )
    else:
        msg = (
            f'⚠ [{sev}] AETHERION Alert\n'
            f'Threat: {threat_class}\n'
            f'Confidence: {confidence*100:.1f}%\n'
            f'GPS: {lat:.4f}°N, {lon:.4f}°E\n'
            f'Responders: {chr(44).join(resp)}'
        )
    
    # Send to appropriate numbers based on severity
    to_numbers = ['+91XXXXXXXXXX']  # Replace with actual numbers
    for num in to_numbers:
        m = client.messages.create(body=msg, from_=TWILIO_FROM, to=num)
        print(f'SMS sent: {m.sid}')

# Test:
# send_tiered_alert('unconscious', 0.92)
print('Alert function ready.')

## 📤 Step 8: Export for Jetson

In [ ]:
# Export to ONNX (Jetson deployment)
best.export(format='onnx', opset=12, simplify=True, half=True)
print('✅ Exported to ONNX')

# Download everything
from google.colab import files
import shutil
shutil.make_archive('/content/aetherion_results', 'zip', '/content/runs/aetherion')
files.download('/content/aetherion_results.zip')
print('✅ Downloaded!')

## 📍 Hyderabad Responder Database Preview

In [ ]:
# Nearest responders from Hussain Sagar (demo)
HYDERABAD_PS = [
    {'name': 'Hussain Sagar PS',        'phone': '+914023232323', 'lat': 17.4239, 'lon': 78.4738},
    {'name': 'Necklace Road PS',         'phone': '+914023456789', 'lat': 17.4205, 'lon': 78.4600},
]

HYDERABAD_HOSPITALS = [
    {'name': 'KIMS Hospital',            'phone': '+914044885000', 'lat': 17.4239, 'lon': 78.4481},
    {'name': 'Apollo Hospitals Jubilee', 'phone': '+914023607777', 'lat': 17.4325, 'lon': 78.4071},
    {'name': 'Nizam Institute (NIMS)',   'phone': '+914023310009', 'lat': 17.3970, 'lon': 78.4600},
]

import math
def dist_km(lat1,lon1,lat2,lon2):
    R=6371
    dlat=math.radians(lat2-lat1); dlon=math.radians(lon2-lon1)
    a=math.sin(dlat/2)**2+math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return round(R*2*math.atan2(a**0.5,(1-a)**0.5),2)

DRONE_LAT, DRONE_LON = 17.4239, 78.4738  # Hussain Sagar

print('Nearest Police Stations:')
for ps in sorted(HYDERABAD_PS, key=lambda x: dist_km(DRONE_LAT,DRONE_LON,x['lat'],x['lon'])):
    d = dist_km(DRONE_LAT,DRONE_LON,ps['lat'],ps['lon'])
    print(f'  {ps["name"]}: {d}km | {ps["phone"]}')

print('\nNearest Hospitals:')
for h in sorted(HYDERABAD_HOSPITALS, key=lambda x: dist_km(DRONE_LAT,DRONE_LON,x['lat'],x['lon'])):
    d = dist_km(DRONE_LAT,DRONE_LON,h['lat'],h['lon'])
    print(f'  {h["name"]}: {d}km | {h["phone"]}')